# 검출 fine-tuning (TASK 6-A / M3)

목표: 시스템 Top-5 **62.9% → ≥81%**

근거 문서 — `docs/metrics_policy.md`, `reports/m2b_spatial_group.md`

## 왜 학습인가
저비용 경로는 전부 소진됐다: 후처리 스윕 개선 0%(M1), GT 없는 조각 재결합 +1.0%p(M2-B).
남은 원인은 **박스 규약 불일치**다 — `8196 P32`는 한 박스인데 옆의 `P32`는 다른 박스이고,
이 구분은 기하 규칙으로 복원할 수 없다. 그러나 **학습은 가능하다.**

## 게이트
- **셀 4** Phase 0 스모크 테스트를 GPU 에서 재실행. 실패하면 여기서 중단(로컬은 CPU 로만 검증했다).
- **셀 6** PP-OCRv4 det baseline 재측정. 지금까지 측정은 PP-OCRv5_server_det(3.x 전용)인데
  학습은 2.x(최대 v4)라, **같은 계열로 기준선을 다시 잡지 않으면 학습 효과와 세대 차가 섞인다.**

## 1. 환경 — GPU 확인 및 버전 고정

In [ ]:
!nvidia-smi

# paddlepaddle-gpu 는 PyPI 에 없다. Baidu 공식 인덱스를 지정해야 한다.
# CUDA 버전은 위 nvidia-smi 출력에 맞춰 cu126 / cu118 등으로 교체할 것.
!python -m pip install -q "paddlepaddle-gpu==3.3.1" -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!python -m pip install -q "paddleocr==3.7.0" rapidfuzz shapely lmdb

import paddle
print("paddle", paddle.__version__,
      "| cuda:", paddle.device.is_compiled_with_cuda(),
      "| devices:", paddle.device.cuda.device_count())
assert paddle.device.is_compiled_with_cuda(), "GPU 빌드가 아니다 — 런타임 유형을 GPU 로 바꿀 것"

## 2. 데이터 — Drive 에서 세션 로컬 디스크로 해제

**Drive 에서 이미지를 직접 읽으면 DataLoader 가 심하게 느려진다.** 이미지는 로컬 디스크(`/content`)에,
체크포인트만 Drive 에 둔다(지시서 Colab 규약).

사전 준비(로컬 PC 에서 1회): `tar -czf label_ocr_images.tar.gz image_set/` 후 Drive 에 업로드.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/label_ocr'
ROOT  = '/content/label_ocr'          # 세션 로컬 (빠름)
!mkdir -p {DRIVE}/experiments {ROOT}

# 이미지 해제 (약 1.4GB)
!tar -xzf {DRIVE}/label_ocr_images.tar.gz -C {ROOT}/
!ls {ROOT}/image_set/*.jpg | wc -l   # 기대: 2340

## 3. 코드 — 우리 저장소 + PaddleOCR 2.10.0

In [ ]:
%cd /content
!git clone -q https://github.com/ocngrn/label_ocr.git /content/label_ocr_repo

# 이미지는 저장소에 없으므로 코드만 ROOT 로 합친다.
# 주의: IPython 의 `!` 는 {..} 를 파이썬 식으로 치환하므로 bash 중괄호 확장을 쓰지 않는다.
import shutil, os
for name in ("src", "tests", "labels", "splits", "snapshots", "configs"):
    shutil.copytree(f"/content/label_ocr_repo/{name}", f"{ROOT}/{name}", dirs_exist_ok=True)
shutil.copy("/content/label_ocr_repo/conftest.py", ROOT)
print(sorted(os.listdir(ROOT)))

# 학습 진입점은 2.x 에만 있다 (3.7 pip 패키지는 추론 전용)
!git clone -q --depth 1 --branch v2.10.0 https://github.com/PaddlePaddle/PaddleOCR.git /content/PaddleOCR

# 사전학습 가중치 (PP-OCRv4 server det)
!mkdir -p /content/weights
!cd /content/weights && wget -q https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_det_server_train.tar && tar xf ch_PP-OCRv4_det_server_train.tar
!ls /content/weights/ch_PP-OCRv4_det_server_train/

## 4. [게이트] Phase 0 스모크 테스트 — GPU 재실행

`docs/framework_decision.md` 4장은 **CPU 에서만** 검증했다. GPU 빌드에서 2.10.0 코드가
도는지 여기서 확인한다. 실패하면 학습에 GPU 시간을 쓰지 않는다.

In [ ]:
import sys, yaml, numpy as np, paddle
sys.path.insert(0, '/content/PaddleOCR')
from ppocr.modeling.architectures import build_model

cfg = yaml.safe_load(open('/content/PaddleOCR/configs/det/ch_PP-OCRv4/ch_PP-OCRv4_det_teacher.yml', encoding='utf-8'))
model = build_model(cfg['Architecture'])

sd = paddle.load('/content/weights/ch_PP-OCRv4_det_server_train/best_accuracy.pdparams')
msd = model.state_dict()
match = [k for k in msd if k in sd and tuple(msd[k].shape) == tuple(sd[k].shape)]
print(f"파라미터 {len(msd)} 중 전이 {len(match)}, 미존재 {len([k for k in msd if k not in sd])}")
model.set_state_dict({k: sd[k] for k in match})

out = model(paddle.to_tensor(np.zeros((1, 3, 640, 640), 'float32')))
print("GPU forward OK ->", {k: tuple(v.shape) for k, v in out.items()})
assert len(match) / len(msd) > 0.9, "가중치 전이율이 낮다 — 템플릿/가중치 조합 확인"

## 5. 크롭 재생성 + 불변 규칙 테스트

크롭(233MB)은 전송하지 않고 여기서 만든다.

In [ ]:
import os
os.chdir(ROOT)
os.makedirs("reports", exist_ok=True)

!python -m src.preprocess.build_labels
!python -m src.matching.build_db
!python -m pytest tests/ -q

## 6. [게이트] PP-OCRv4 det baseline 재측정

**이 셀을 건너뛰면 학습 효과를 측정할 수 없다.** 기존 baseline 은 PP-OCRv5_server_det 로 쟀는데
학습은 v4 계열이므로, 같은 계열의 기준선을 여기서 확보한다.

In [ ]:
import os, json
os.chdir(ROOT)
from src.eval import detect_baseline

m = detect_baseline.evaluate("PP-OCRv4_server_det")     # 학습과 같은 계열로 기준선 확보
print(json.dumps({k: m[k] for k in ("overall", "plane", "curved")}, ensure_ascii=False, indent=2))
json.dump(m, open("reports/baseline_det_v4.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("검출 recall(평면):", f"{m['plane']['recall']*100:.1f}%",
      "-> 시스템 Top-5 추정:", f"{m['plane']['recall']*0.856*100:.1f}%")

## 7. 학습

`save_epoch_step=5` + `checkpoints` 로 세션이 끊겨도 이어서 학습한다.
**세션이 끊기면 셀 1~3 을 다시 돌린 뒤 아래 `RESUME` 만 켜고 재실행하면 된다.**

In [ ]:
import os, subprocess
os.chdir(ROOT)
RESUME = None     # 이어서 학습할 때: f"{DRIVE}/experiments/det_v4_server/latest"

cmd = ["python", "-m", "src.configs.build_det_config",
       "--template", "/content/PaddleOCR/configs/det/ch_PP-OCRv4/ch_PP-OCRv4_det_teacher.yml",
       "--out", f"{ROOT}/configs/det_ppocrv4_server.yml",
       "--root", ROOT,
       "--weights", "/content/weights/ch_PP-OCRv4_det_server_train/best_accuracy",
       "--save-dir", f"{DRIVE}/experiments/det_v4_server",
       "--epochs", "50", "--batch-size", "8"]
if RESUME:
    cmd += ["--checkpoints", RESUME]
print(subprocess.run(cmd, capture_output=True, text=True).stdout)

os.chdir("/content/PaddleOCR")
!python tools/train.py -c {ROOT}/configs/det_ppocrv4_server.yml

## 8. 평가 — M3 판정

두 층위로 나눠 본다(`docs/metrics_policy.md` 4장의 L2 / L3).

- **L2 모듈 KPI**: 검출 Hmean/recall — 학습이 먹혔는가
- **L3 시스템 KPI**: end-to-end Top-5 — **유일한 판정 기준**, 목표 ≥81%

In [ ]:
# L2 — 2.x 자체 평가 (val)
%cd /content/PaddleOCR
!python tools/eval.py -c {ROOT}/configs/det_ppocrv4_server.yml     -o Global.checkpoints={DRIVE}/experiments/det_v4_server/best_accuracy

# 추론 모델로 export (end-to-end 에 필요)
!python tools/export_model.py -c {ROOT}/configs/det_ppocrv4_server.yml     -o Global.pretrained_model={DRIVE}/experiments/det_v4_server/best_accuracy        Global.save_inference_dir={DRIVE}/experiments/det_v4_server/inference

### L3 재측정에 대한 주의

`src/eval/end_to_end.py` 는 `paddleocr` 3.7 의 `TextDetection` 을 쓴다. 학습 결과는 2.x 로
export 한 추론 모델이므로, **검출기를 교체할 어댑터가 필요하다.** 이 노트북 범위 밖이며,
L2 개선이 확인된 뒤 로컬에서 어댑터를 붙여 재측정한다.

임시로는 학습 전후 **검출 recall** 만 비교해 방향을 확인하고,
`시스템 Top-5 ≈ 검출 recall × 85.6%(조건부 인식)` 로 추정한다.